# [16.1] Exact Shapley on Ground-Truth Games - Exercises

Build exact Shapley values on complete finite games. The real CUDA neural-game evidence is pinned in `verification_report.json`.

In [ ]:
from collections.abc import Callable, Mapping
from dataclasses import dataclass
import itertools
import math
import sys
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part1_exact_shapley_ground_truth_games"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_exact_shapley_ground_truth_games.tests as tests

GT_TIER = "GT-0"
EXERCISE_ID = "16_1_exact_shapley_on_ground_truth_games"
EXPECTED_RUNTIME = "35-45 minutes for exercises; about 2 minutes for the CUDA neural-game preflight"
REQUIRES_GPU = True

Coalition = frozenset[int]

In [ ]:
@dataclass(frozen=True)
class ShapleyEfficiencyReport:
    shapley_sum: float
    total_value_delta: float
    efficiency_error: float
    satisfies_efficiency: bool


@dataclass(frozen=True)
class PermutationParityReport:
    max_abs_error: float
    matches_exact: bool


@dataclass(frozen=True)
class InteractionGapReport:
    shapley_total: float
    leave_one_out_total: float
    overcount: float
    detects_interaction_overcount: bool

## Coalition Tables

Enumerate every coalition and reject incomplete coalition-value tables.

In [ ]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    raise NotImplementedError()


def normalize_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> dict[Coalition, float]:
    raise NotImplementedError()


def coalition_values_from_function(
    num_players: int,
    value_fn: Callable[[Coalition], float],
) -> dict[Coalition, float]:
    raise NotImplementedError()


tests.test_all_coalitions_enumerates_the_power_set(all_coalitions)

## Exact Shapley Values

Implement the weighted marginal-effect formula and recover additive feature weights.

In [ ]:
def exact_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    raise NotImplementedError()


def additive_game(weights: t.Tensor) -> dict[Coalition, float]:
    raise NotImplementedError()


tests.test_additive_game_and_exact_shapley_recover_weights(
    additive_game,
    exact_shapley_values,
)
tests.test_exact_shapley_requires_a_complete_coalition_table(exact_shapley_values)

## Efficiency

Check that exact Shapley values sum to `v(full) - v(empty)`.

In [ ]:
def conjunction_game(num_players: int) -> dict[Coalition, float]:
    raise NotImplementedError()


def shapley_efficiency_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    tolerance: float = 1e-9,
) -> ShapleyEfficiencyReport:
    raise NotImplementedError()


tests.test_conjunction_game_splits_symmetric_credit_and_checks_efficiency(
    conjunction_game,
    exact_shapley_values,
    shapley_efficiency_report,
)

## Permutation Parity

Compare closed-form exact Shapley values to exact averaging over feature orderings.

In [ ]:
def permutation_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    raise NotImplementedError()


def permutation_parity_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    tolerance: float = 1e-9,
) -> PermutationParityReport:
    raise NotImplementedError()


tests.test_permutation_parity_report_matches_exact_formula(
    conjunction_game,
    permutation_parity_report,
)

## Interaction Failures

Show that full-minus-ablated leave-one-out scores overcount a two-player AND game.

In [ ]:
def leave_one_out_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    raise NotImplementedError()


def interaction_gap_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    min_overcount: float = 0.5,
) -> InteractionGapReport:
    raise NotImplementedError()


tests.test_interaction_gap_report_catches_leave_one_out_overcount(
    conjunction_game,
    interaction_gap_report,
)

## Notebook Contract

Assemble the deterministic toy checks recorded by the verification report.

In [ ]:
def additive_smoke_test() -> dict:
    weights = t.tensor([1.0, 2.0, -0.5])
    values = additive_game(weights)
    shapley = exact_shapley_values(values, num_players=3)
    return {
        "shapley": shapley.tolist(),
        "efficiency": shapley_efficiency_report(values, num_players=3).__dict__,
    }


def conjunction_smoke_test() -> dict:
    values = conjunction_game(3)
    shapley = exact_shapley_values(values, num_players=3)
    return {
        "shapley": shapley.tolist(),
        "efficiency": shapley_efficiency_report(values, num_players=3).__dict__,
    }


def permutation_parity_smoke_test() -> dict:
    values = conjunction_game(3)
    return permutation_parity_report(values, num_players=3).__dict__


def interaction_failure_smoke_test() -> dict:
    values = conjunction_game(2)
    return interaction_gap_report(values, num_players=2, min_overcount=0.5).__dict__


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "additive": additive_smoke_test(),
        "conjunction": conjunction_smoke_test(),
        "permutation_parity": permutation_parity_smoke_test(),
        "interaction_failure": interaction_failure_smoke_test(),
    }


tests.test_additive_smoke_test(additive_smoke_test)
tests.test_conjunction_smoke_test(conjunction_smoke_test)
tests.test_permutation_parity_smoke_test(permutation_parity_smoke_test)
tests.test_interaction_failure_smoke_test(interaction_failure_smoke_test)
tests.test_notebook_contract(run_smoke_test)

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
